# Boundaries: a pinned LBD release and raking inputs

The loaders read real releases from `paths.yaml`; this notebook writes a two-country release
of its own in a temporary folder next to the notebook so it runs anywhere. See
`docs/boundaries.md`.

In [ ]:
import shutil
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import box

from idd_spatial_analysis.boundaries import (
    AdminRelease,
    copy_release,
    load_admin,
    load_hierarchy,
    load_raking_shapes,
)
from idd_spatial_analysis.constants import WGS84, WORLD_CYLINDRICAL_EQUAL_AREA

WORK = Path.cwd() / "_boundaries_vignette_tmp"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

In [ ]:
# A synthetic release: lbd_standard_admin_{0,1,2}[_simplified].shp under <root>/<YYYY_MM_DD>/.
root = WORK / "admin_shapefiles"
release_dir = root / "2024_07_29"
release_dir.mkdir(parents=True)
adm0 = gpd.GeoDataFrame(
    {
        "ADM0_CODE": [1, 2],
        "ADM0_NAME": ["Aland", "Bland"],
        "loc_id": pd.array([10, None], dtype="Int64"),
    },
    geometry=[box(0, 0, 10, 10), box(10, 0, 20, 10)],
    crs=WGS84,
)
adm1 = gpd.GeoDataFrame(
    {
        "ADM0_CODE": [1, 1, 2],
        "ADM1_CODE": [11, 12, 21],
        "ADM1_NAME": ["A-north", "A-south", "B-all"],
    },
    geometry=[box(0, 5, 10, 10), box(0, 0, 10, 5), box(10, 0, 20, 10)],
    crs=WGS84,
)
adm2 = gpd.GeoDataFrame(
    {
        "ADM0_CODE": [1, 1, 1, 2],
        "ADM1_CODE": [11, 11, 12, 21],
        "ADM2_CODE": [111, 112, 121, 211],
        "ADM2_NAME": ["A-nw", "A-ne", "A-s", "B"],
    },
    geometry=[box(0, 5, 5, 10), box(5, 5, 10, 10), box(0, 0, 10, 5), box(10, 0, 20, 10)],
    crs=WGS84,
)
for level, gdf in enumerate((adm0, adm1, adm2)):
    for suffix in ("", "_simplified"):
        gdf.to_file(release_dir / f"lbd_standard_admin_{level}{suffix}.shp")
sorted(p.name for p in release_dir.iterdir())[:6]

In [ ]:
# Pin the release explicitly; nothing picks "the latest" for you.
release = AdminRelease(root, "2024_07_29", simplified=False)
print(release.path(2))
print(load_admin(release, 2).dtypes)

In [ ]:
# Subset by country at read time, add columns, reproject.
only_a = load_admin(release, 2, countries=[1], crs=WORLD_CYLINDRICAL_EQUAL_AREA)
print(only_a[["ADM0_CODE", "ADM1_CODE", "ADM2_CODE", "ADM2_NAME"]])
print(only_a.crs)
print(load_admin(release, 0, columns=["loc_id"])["loc_id"])

In [ ]:
# Copy a release where the archive mount is not visible (the CLI does the same from paths.yaml).
copied = copy_release(release, WORK / "project_store")
print(copied.directory)
print(len(load_admin(copied, 1)), "admin-1 units from the copy")

In [ ]:
# Raking inputs: most-detailed shapes of a hierarchy and its parent table.
raking = WORK / "raking"
(raking / "gbd-inputs").mkdir(parents=True)
shapes = gpd.GeoDataFrame(
    {"location_id": [100, 101, 102]},
    geometry=[box(0, 0, 10, 10), box(0, 0, 5, 10), box(5, 0, 10, 10)],
    crs=WGS84,
)
shapes.to_parquet(raking / "shapes_gbd_2023.parquet", write_covering_bbox=True)
pd.DataFrame(
    {"location_id": [100, 101, 102], "year_id": [2023] * 3, "most_detailed": [0, 1, 1]}
).to_parquet(raking / "population_gbd_2023.parquet")
pd.DataFrame(
    {"location_id": [100, 101, 102], "parent_id": [0, 100, 100], "level": [0, 1, 1]}
).to_parquet(raking / "gbd-inputs" / "hierarchy_gbd_2023.parquet")
print(load_raking_shapes(raking, "gbd_2023"))
print(load_raking_shapes(raking, "gbd_2023", bounds=(6.0, 1.0, 7.0, 2.0)))
print(load_hierarchy(raking, "gbd_2023").dtypes)

In [ ]:
shutil.rmtree(WORK)